# Exp6 × conversion cost — does translator cost order the ε you already measured?

**Question.** Exp8's hypothesis is that conversion cost between two renderings predicts how much
sharing costs. Exp6 already measured ε (shared-tree CE − disjoint-grammar CE) for L_B and L_C at
d_model=96. This notebook measures the conversion cost `a(g)` of L_B→L_A and L_C→L_A on the **same
grammar, same architecture (12 layers, 12 heads, d=96)**, and checks whether it orders ε correctly.
No capacity window is needed: both quantities are measured where the ε signal already exists.

## Frozen BEFORE any result is read

**ε already observed** (copied from the Exp6 notebook, d_model=96, `eps_pure = CE_shared − CE_disjoint`):

| run | ε_A | ε_B | ε_C | ε_C − ε_B |
|---|---|---|---|---|
| seed 1 (119k steps, step-matched) | −0.0045 | +0.0227 | +0.0603 | +0.0376 |
| seed 7 (90k steps) | −0.0097 | +0.0458 | +0.0644 | +0.0186 |

Observed ordering in both seeds: **ε_C > ε_B**.

**Prediction.** If conversion cost orders ε, then `a(L_C) > a(L_B)`.
This is not obvious in advance: L_C→L_A is a whole-string reversal (needs length-aware routing),
while L_B→L_A needs the tree structure (reverse only at levels 4 and 5). If the translator finds
L_B harder, the hypothesis fails at this capacity.

**Decision rule (applied by the last cell, not by hand).**
- `a(g) = AUC_g − AUC_identity` per seed, paired by seed (same trees for identity, L_B, L_C).
- PASS = `a(L_C) > a(L_B)` in **every** seed. FAIL = `a(L_B) > a(L_C)` in every seed. Otherwise INCONCLUSIVE.
- INVALID if identity heldout accuracy < 0.99 in any seed (the instrument check from Exp8's V1).
- Any L_B / L_C curve still improving at the end is flagged: treat that verdict as inconclusive and raise `TRANS_STEPS`.
- If both final heldout CEs are < 0.01, a PASS is labelled **SPEED-ONLY** (the difference is learning speed, not an asymptotic ceiling, which is weaker support for a theory about circuit cost rather than SGD time).
- No seed-hunting, no changing the rule after seeing the numbers. A FAIL is reported as a FAIL.

In [1]:
import os, json, time, random, contextlib
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# ================================ CONFIG ================================
QUICK_TEST = False       # True = tiny shakedown (CPU-able, writes to a separate folder). Flip to False for the real run.

DEVICE_STR = "cuda:2"    # check nvidia-smi before assuming this is free
DEVICE = torch.device(DEVICE_STR if torch.cuda.is_available() else "cpu")

TR_D, TR_N_LAYER, TR_N_HEAD = 96, 12, 12   # same arch/capacity as Exp6's joint models
TRANS_STEPS = 30_000
SEEDS = [0, 1, 2]
BATCH, LR, WIN = 32, 3e-3, 500
POOL_N, EVAL_N = 100_000, 1_000
MAX_LEAVES = 320         # keeps 94% of Exp6's tree-size distribution (mean 237, max 406 leaves)
ROOT = "./exp6_conv_cost"

if QUICK_TEST:
    TR_D, TR_N_LAYER, TR_N_HEAD = 16, 2, 2
    TRANS_STEPS, SEEDS, WIN = 600, [0], 100
    POOL_N, EVAL_N = 3_000, 200
    ROOT = "./exp6_conv_cost_QUICK"

ROW_LEN = 2 * MAX_LEAVES + 2

# Exp6 grammar settings (unchanged)
LEVELS, NT_SIZE, TERM_SIZE = 6, 4, 4
MIN_DEGREE, MAX_DEGREE, RULE_LEN_CHOICES = 2, 4, [2, 3]
GRAMMAR_SEED = 0

# renderings; the translator always decodes canonical (= L_A) from the g-rendering
LANGS = {"identity": set(), "L_B": {4, 5}, "L_C": set(range(1, LEVELS + 1))}
TRAIN_ORDER = ["identity", "L_B", "L_C"]

# translator vocab: terminals 0-3, one BOS tag per rendering, SEP
BOS = {"identity": 4, "L_B": 5, "L_C": 6}
SEP = 7
VOCAB = 8

print(f"device={DEVICE}  QUICK_TEST={QUICK_TEST}  d={TR_D} layers={TR_N_LAYER} heads={TR_N_HEAD}  steps={TRANS_STEPS}  seeds={SEEDS}")

device=cuda:2  QUICK_TEST=False  d=96 layers=12 heads=12  steps=30000  seeds=[0, 1, 2]


## Grammar (copied unchanged from Exp6) and tree-size check

In [2]:
from collections import defaultdict

def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen:
                    continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def linearize(node, reverse_levels):
    if isinstance(node, int):
        return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize(c, reverse_levels))
    return tokens

rules, _ = build_grammar(GRAMMAR_SEED)

# sanity: L_C must be exactly reverse(L_A); L_B must differ from both
_rng = random.Random(123)
_t = sample_tree(rules, _rng)
_a, _b, _c = (linearize(_t, LANGS[k]) for k in ("identity", "L_B", "L_C"))
assert _c == list(reversed(_a)), "L_C should equal reverse(L_A)"
assert _b != _a and _b != _c, "L_B should differ from both L_A and L_C"
assert sorted(_a) == sorted(_b) == sorted(_c), "same multiset of leaves across renderings"

# what fraction of trees survives the MAX_LEAVES cap (distribution shift vs Exp6's training data)?
_lens = np.array([len(linearize(sample_tree(rules, _rng), set())) for _ in range(5000)])
print(f"leaves: mean={_lens.mean():.0f} median={np.median(_lens):.0f} max={_lens.max()}  "
      f"fraction kept at MAX_LEAVES={MAX_LEAVES}: {(_lens <= MAX_LEAVES).mean():.3f}")
print("grammar sanity checks passed")

leaves: mean=239 median=246 max=409  fraction kept at MAX_LEAVES=320: 0.943
grammar sanity checks passed


## Data: translator rows

Row = `[BOS_g, g(z)..., SEP, canonical(z)...]`, loss only on the canonical tokens (same format as Exp8).
One pool of trees per data seed is rendered under all three languages, so identity / L_B / L_C see the
**same trees** within a seed (paired comparison). Held-out trees come from a fixed separate seed.

In [3]:
_POOL_CACHE = {}

def build_pool(data_seed, n):
    """n trees (<= MAX_LEAVES leaves), each stored as int8 arrays under all three renderings."""
    key = (data_seed, n)
    if key in _POOL_CACHE:
        return _POOL_CACHE[key]
    rng = random.Random(data_seed)
    out = {name: [] for name in LANGS}
    while len(out["identity"]) < n:
        t = sample_tree(rules, rng)
        a = linearize(t, LANGS["identity"])
        if len(a) > MAX_LEAVES:
            continue
        out["identity"].append(np.array(a, np.int8))
        for name in ("L_B", "L_C"):
            out[name].append(np.array(linearize(t, LANGS[name]), np.int8))
    _POOL_CACHE[key] = out
    return out

def make_rows(pool, g):
    n = len(pool["identity"])
    rows = np.zeros((n, ROW_LEN), np.int8)
    ostart = np.zeros(n, np.int64)
    rlen = np.zeros(n, np.int64)
    for k in range(n):
        gz, c = pool[g][k], pool["identity"][k]
        L = len(c)
        assert len(gz) == L
        rows[k, 0] = BOS[g]
        rows[k, 1:1 + L] = gz
        rows[k, 1 + L] = SEP
        rows[k, 2 + L:2 + 2 * L] = c
        ostart[k] = L + 2          # index of the first canonical token
        rlen[k] = 2 * L + 2
    assert rlen.max() <= ROW_LEN
    return rows, ostart, rlen

# quick self-check of the row format
_p = build_pool(1, 5)
_r, _o, _l = make_rows(_p, "L_C")
_k = 0
assert _r[_k, 0] == BOS["L_C"] and _r[_k, _o[_k] - 1] == SEP
assert list(_r[_k, _o[_k]:_l[_k]]) == list(_p["identity"][_k])
assert list(_r[_k, 1:_o[_k] - 1]) == list(_p["L_C"][_k])
print("row format OK")

row format OK


## Model (copied unchanged from Exp6: rotary GPT, untied head)

In [4]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))

_m = TinyRotaryGPT(VOCAB, TR_D, TR_N_LAYER, TR_N_HEAD, ROW_LEN + 1)
print(f"model OK, {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params at d={TR_D}")
del _m

model OK, 1.34M params at d=96


## Translator training (fixed step budget, no early stopping)

Same loop as Exp8's `train_translator`: AdamW, lr 3e-3, batch 32, loss masked to the canonical side,
window-averaged loss curve, `a(g)` later read as the AUC difference to identity. Fixed budget on purpose
(the Exp6 lesson: adaptive "converged" flags and mismatched step counts distort comparisons).
Results are saved per `(g, seed)` and the function skips anything already done, so the launch cell is safe to rerun.

In [5]:
def train_translator(g, seed):
    path = f"{ROOT}/translator/{g}_d{TR_D}_s{seed}.json"
    if os.path.exists(path):
        print("done already:", path)
        return json.load(open(path))
    os.makedirs(os.path.dirname(path), exist_ok=True)

    pool = build_pool(5000 + seed, POOL_N)                     # same trees for identity / L_B / L_C at this seed
    rows, ostart, rlen = make_rows(pool, g)
    R = torch.from_numpy(rows).to(DEVICE)
    OS = torch.from_numpy(ostart).to(DEVICE)
    RL = torch.from_numpy(rlen).to(DEVICE)

    torch.manual_seed(seed)
    model = TinyRotaryGPT(VOCAB, TR_D, TR_N_LAYER, TR_N_HEAD, ROW_LEN + 1).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, fused=(DEVICE.type == "cuda"))
    ar = torch.arange(ROW_LEN - 1, device=DEVICE)
    def masked(os_, rl):   # predict token index ar+1; keep canonical side only
        return ((ar[None, :] + 1 >= os_[:, None]) & (ar[None, :] + 1 < rl[:, None])).float()
    ac = (lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16)) if DEVICE.type == "cuda" else None

    curve, acc, cnt = [], 0.0, 0
    t0 = time.time()
    for step in range(TRANS_STEPS):
        idx = torch.randint(0, R.shape[0], (BATCH,), device=DEVICE)
        b = R[idx].long()
        x, y = b[:, :-1], b[:, 1:]
        with (ac() if ac else contextlib.nullcontext()):
            logits = model(x)
        ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="none").view(BATCH, -1)
        m = masked(OS[idx], RL[idx])
        loss = (ce * m).sum() / m.sum()
        opt.zero_grad(); loss.backward(); opt.step()
        acc += loss.item(); cnt += 1
        if step == 199:
            dt = (time.time() - t0) / 200
            print(f"  [{g} s{seed}] ~{dt*1000:.0f} ms/step -> projected {dt*TRANS_STEPS/60:.0f} min for this run", flush=True)
        if cnt == WIN:
            curve.append(acc / cnt); acc, cnt = 0.0, 0
            if len(curve) % 10 == 0:
                print(f"  [{g} s{seed}] step {step+1:>6d} out-CE {curve[-1]:.4f}", flush=True)

    # held-out trees (fixed separate seed), same rows format
    epool = build_pool(777777, EVAL_N)
    erows, eos, erl = make_rows(epool, g)
    model.eval()
    tot = tok = hit = 0.0
    with torch.no_grad():
        for i in range(0, EVAL_N, 32):
            bb = torch.from_numpy(erows[i:i + 32].astype(np.int64)).to(DEVICE)
            x, y = bb[:, :-1], bb[:, 1:]
            logits = model(x)
            ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="none").view(bb.shape[0], -1)
            mm = masked(torch.from_numpy(eos[i:i + 32]).to(DEVICE), torch.from_numpy(erl[i:i + 32]).to(DEVICE))
            tot += (ce * mm).sum().item(); tok += mm.sum().item()
            hit += ((logits.argmax(-1) == y).float() * mm).sum().item()

    res = dict(g=g, d=TR_D, seed=seed, steps=TRANS_STEPS, win=WIN, curve=curve,
               auc=float(np.mean(curve)), final_train=float(np.mean(curve[-4:])),
               heldout_ce=tot / tok, heldout_acc=hit / tok)
    json.dump(res, open(path, "w"))
    print(json.dumps({k: v for k, v in res.items() if k != "curve"}), flush=True)
    return res

## Run

Order is per seed (identity, L_B, L_C), so a partial run still gives complete paired results for the finished seeds.
Watch the `ms/step` projection on the first run before leaving it. With `QUICK_TEST = True` this is a shakedown only:
it checks the pipeline runs, and its numbers mean nothing.

In [6]:
for seed in SEEDS:
    for g in TRAIN_ORDER:
        train_translator(g, seed)
print("\nall requested runs finished")

  [identity s0] ~32 ms/step -> projected 16 min for this run
  [identity s0] step   5000 out-CE 0.0043
  [identity s0] step  10000 out-CE 0.0023
  [identity s0] step  15000 out-CE 0.0010
  [identity s0] step  20000 out-CE 0.0009
  [identity s0] step  25000 out-CE 0.0010
  [identity s0] step  30000 out-CE 0.0009
{"g": "identity", "d": 96, "seed": 0, "steps": 30000, "win": 500, "auc": 0.06685104478256901, "final_train": 0.0008960611185912057, "heldout_ce": 0.0005709836827529762, "heldout_acc": 0.9998201415742751}
  [L_B s0] ~33 ms/step -> projected 17 min for this run
  [L_B s0] step   5000 out-CE 0.1343
  [L_B s0] step  10000 out-CE 0.0676
  [L_B s0] step  15000 out-CE 0.0490
  [L_B s0] step  20000 out-CE 0.0380
  [L_B s0] step  25000 out-CE 0.0314
  [L_B s0] step  30000 out-CE 0.0284
{"g": "L_B", "d": 96, "seed": 0, "steps": 30000, "win": 500, "auc": 0.11223328475126376, "final_train": 0.028446582552045584, "heldout_ce": 0.029555638672024504, "heldout_acc": 0.9879494854764321}
  [L_C s

## Verdict (applies the rule frozen at the top)

In [8]:
# ε observed in Exp6 (frozen before this test; see header)
EPS_OBS = {
    "seed1": {"L_A": -0.0045, "L_B": 0.0227, "L_C": 0.0603},
    "seed7": {"L_A": -0.0097, "L_B": 0.0458, "L_C": 0.0644},
}

def load(g, seed):
    p = f"{ROOT}/translator/{g}_d{TR_D}_s{seed}.json"
    return json.load(open(p)) if os.path.exists(p) else None

def still_improving(curve):
    if len(curve) < 12:
        return None
    prev, last = np.mean(curve[-6:-3]), np.mean(curve[-3:])
    return bool(prev > 0.01 and (prev - last) > 0.10 * prev)

seeds_done = [s for s in SEEDS if all(load(g, s) for g in TRAIN_ORDER)]
print(f"seeds with all three runs: {seeds_done}")
assert seeds_done, "no complete seed yet, run the cell above first"

rows_out, flags = [], []
for s in seeds_done:
    R = {g: load(g, s) for g in TRAIN_ORDER}
    aB = R["L_B"]["auc"] - R["identity"]["auc"]
    aC = R["L_C"]["auc"] - R["identity"]["auc"]
    rows_out.append(dict(seed=s, aB=aB, aC=aC, R=R))
    if R["identity"]["heldout_acc"] < 0.99:
        flags.append(f"seed {s}: identity heldout_acc {R['identity']['heldout_acc']:.4f} < 0.99 (instrument invalid)")
    for g in ("L_B", "L_C"):
        if still_improving(R[g]["curve"]):
            flags.append(f"seed {s}: {g} curve still improving at the end (raise TRANS_STEPS)")

print(f"\n{'seed':>4} {'g':>9} {'AUC':>8} {'a(g)':>8} {'heldout_CE':>11} {'heldout_acc':>12}  last 6 curve points")
for r in rows_out:
    for g in TRAIN_ORDER:
        x = r["R"][g]
        a = "" if g == "identity" else f"{(r['aB'] if g=='L_B' else r['aC']):8.4f}"
        print(f"{r['seed']:>4} {g:>9} {x['auc']:8.4f} {a:>8} {x['heldout_ce']:11.5f} {x['heldout_acc']:12.4f}  "
              f"{[round(v,4) for v in x['curve'][-6:]]}")

diffs = np.array([r["aC"] - r["aB"] for r in rows_out])
print(f"\na(L_C) - a(L_B) per seed: {np.round(diffs, 4).tolist()}")
if len(diffs) > 1:
    print(f"mean {diffs.mean():.4f}  SE {diffs.std(ddof=1)/np.sqrt(len(diffs)):.4f}  (n={len(diffs)} seeds)")
print("observed eps_C - eps_B in Exp6:", {k: round(v["L_C"] - v["L_B"], 4) for k, v in EPS_OBS.items()}, " -> prediction: a(L_C) > a(L_B)")

if any("instrument invalid" in f for f in flags):
    verdict = "INVALID (identity did not converge, a(g) cannot be read)"
elif (diffs > 0).all():
    verdict = "PASS"
elif (diffs < 0).all():
    verdict = "FAIL"
else:
    verdict = "INCONCLUSIVE (sign differs across seeds)"

at_floor = all(r["R"][g]["heldout_ce"] < 0.01 for r in rows_out for g in ("L_B", "L_C"))
if verdict == "PASS" and at_floor:
    verdict += " -- SPEED-ONLY (both translators reach ~0 CE; the gap is learning speed, not an asymptotic ceiling)"
if any("still improving" in f for f in flags) and not verdict.startswith("INVALID"):
    verdict += " -- UNDERTRAINED, treat as inconclusive"
if len(rows_out) < 3:
    verdict += f" -- PROVISIONAL (only {len(rows_out)} seed(s); the rule needs 3)"

print("\nflags:", flags if flags else "none")
print("\nVERDICT:", verdict)

seeds with all three runs: [0, 1, 2]

seed         g      AUC     a(g)  heldout_CE  heldout_acc  last 6 curve points
   0  identity   0.0669              0.00057       0.9998  [0.0007, 0.0007, 0.0009, 0.001, 0.0009, 0.0009]
   0       L_B   0.1122   0.0454     0.02956       0.9879  [0.0296, 0.0297, 0.0284, 0.0288, 0.0282, 0.0284]
   0       L_C   0.0665  -0.0004     0.00011       1.0000  [0.0002, 0.0003, 0.0042, 0.0001, 0.0002, 0.0001]
   1  identity   0.0673              0.00048       0.9999  [0.0008, 0.001, 0.0041, 0.0005, 0.0004, 0.0009]
   1       L_B   0.1308   0.0634     0.03252       0.9868  [0.035, 0.0346, 0.0341, 0.0338, 0.0338, 0.0326]
   1       L_C   0.0582  -0.0091     0.00015       1.0000  [0.0003, 0.0005, 0.0006, 0.0032, 0.0001, 0.0003]
   2  identity   0.0441              0.00013       1.0000  [0.0033, 0.0004, 0.0006, 0.0008, 0.0036, 0.0009]
   2       L_B   0.1325   0.0884     0.04014       0.9839  [0.0383, 0.0379, 0.0383, 0.0368, 0.0371, 0.0365]
   2       L_C   0.066

## How to read it, and what it can't show

- **PASS** means conversion cost ordered ε correctly on the one comparison available. With three languages there is a single binary comparison (L_B vs L_C), so luck alone passes half the time. Treat a PASS as mild support, not confirmation.
- **FAIL** is more informative: both quantities are measured where they are valid, so imprecision can't be blamed.
- **SPEED-ONLY** PASS: at d=96 both translators may reach ~0 CE, in which case `a(g)` reflects how fast SGD finds the circuit. Your theory is about circuit cost, not SGD time, so this is the weak form of support.
- ε from Exp6 is itself not fully converged (joint losses were still drifting down ~0.0005 per 5k steps), and ε_B varied a lot across the two seeds. The ordering is stable; the magnitudes are not.
- Trees are capped at 320 leaves (94% kept), so the translator sees slightly smaller trees than the joint models did.
- To make this a real correlation instead of a single comparison, you need more linearizations *with their own ε runs* (levels {5}, {4,5,6}, {2,3}, ...). Write the predicted ordering from `a(g)` first, then run the joint grid.